In [7]:
import numpy as np
from Environment import DRLControllerEnv
import config

In [3]:
class Normalizer:
    def __init__(self, obs_length):
        self.mean = np.zeros(obs_length)
        self.n = np.zeros(obs_length)
        self.sos_diff = np.zeros(obs_length)
        self.var = np.zeros(obs_length)

    def update_statistics(self, obs):
        self.n += 1
        last_mean = self.mean.copy()
        self.mean += (obs - self.mean) / self.n
        self.sos_diff += (obs - last_mean) * (obs - self.mean)
        self.var = (self.sos_diff / self.n).clip(min=1e-2)

    def normalize(self, obs):
        self.update_statistics(obs)
        obs_no_mean = obs - self.mean
        obs_std = np.sqrt(self.var)
        return obs_no_mean / obs_std

In [4]:
class ARSAgent:
    def __init__(
        self,
        env: DRLControllerEnv,
        rand_directions: int = 2,         # from training notebook
        best_directions: int = 1,         # from training notebook
        learning_rate: float = 0.002,     # α from training notebook
        exploration_noise: float = 0.02,  # ν from training notebook
        n_simulations: int = 2,           # number of ARS simulations
        rollout_length: int = 60          # timesteps per episode
    ):
        # ARS hyperparams
        assert best_directions <= rand_directions, "best_directions must be <= rand_directions"
        self.rand_dir = rand_directions
        self.best_dir = best_directions
        self.alpha = learning_rate
        self.nu = exploration_noise
        self.n_simulations = n_simulations
        self.rollout_length = rollout_length

        # Environment
        self.env = env
        self.obs_dim = self.env.observation_space.shape[0]
        self.act_dim = self.env.action_space.shape[0]

        # Normalizer for states
        self.normalizer = Normalizer(self.obs_dim)

        # Linear policy initialization
        from LinearPolicy import LinearPolicy
        self.policy = LinearPolicy(state_shape=self.env.observation_space.shape,
                                   action_shape=self.env.action_space.shape)
        self.theta = self.policy.get_policy().copy()

    def _rollout(self, theta):
        obs = self.env.reset()
        done = False
        total_reward = 0.0
        steps = 0
        while not done:
            norm_obs = self.normalizer.normalize(obs)
            action = self.policy.compute_action(norm_obs, theta)
            obs, reward, done, _ = self.env.step(action)
            total_reward += reward
            steps += 1
            if steps >= self.rollout_length:
                break
        return total_reward

    def train(self):
        for sim in range(self.n_simulations):
            # sample random directions
            deltas = [np.random.randn(*self.theta.shape) for _ in range(self.rand_dir)]

            positive_rewards = []
            negative_rewards = []

            # evaluate each direction
            for delta in deltas:
                # average multiple rollouts per direction if desired
                rp = self._rollout(self.theta + self.nu * delta)
                rn = self._rollout(self.theta - self.nu * delta)
                positive_rewards.append(rp)
                negative_rewards.append(rn)

            # select best directions
            scores = np.array([max(rp, rn) for rp, rn in zip(positive_rewards, negative_rewards)])
            idxs = np.argsort(scores)[-self.best_dir:]

            # compute update step
            reward_std = np.array(positive_rewards + negative_rewards).std()
            update = np.zeros_like(self.theta)
            for idx in idxs:
                update += (positive_rewards[idx] - negative_rewards[idx]) * deltas[idx]

            self.theta += (self.alpha / (self.best_dir * reward_std)) * update
            self.policy.update_policy(self.theta)

            # report progress
            avg_reward = self._rollout(self.theta)
            print(f"Simulation {sim+1}/{self.n_simulations}: AvgReward={avg_reward:.2f}")

        return self.theta




In [8]:
# Usage example
if __name__ == '__main__':
    env = DRLControllerEnv(
        simulation_time=config.SIMULATION_TIME,
        time_step=config.TIME_STEP
    )
    agent = ARSAgent(env)
    optimal_theta = agent.train()

ModuleNotFoundError: No module named 'LinearPolicy'